# Pipeline SFT — Logic (trắc nghiệm)

Code trong `Logic_Based_Educational_Queries_Project/`: `src/services/`, `src/models/logic_model/`.

**Kaggle:** Add Data (full repo có `src/`) hoặc `git clone` + `%cd`. Tuỳ chọn: `LOGIC_PROJECT_ROOT` trong Environment.

- Split 8:1:1 theo `record_id`; checkpoint theo **`eval_accuracy`**.


## 0. Dependency (chạy khi cần)

In [ ]:
# Kaggle: bỏ comment dòng pip nếu thiếu package
%pip install -q "trl>=0.16.0" peft accelerate transformers datasets huggingface_hub python-dotenv scikit-learn bitsandbytes


## 1. Secrets (Kaggle) / token HF

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    
    HF_Token = UserSecretsClient().get_secret("HF_Token")
except Exception:
    HF_Token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN") or ""

## 2. Bootstrap (Kaggle + local)


In [ ]:
import os, sys
from pathlib import Path

NEST = "Logic_Based_Educational_Queries_Project"


def logic_repo_root() -> Path:
    for key in ("LOGIC_PROJECT_ROOT", "EXACT_PROJECT_ROOT"):
        v = os.environ.get(key, "").strip()
        if v:
            p = Path(v).expanduser().resolve()
            for cand in (p, p / NEST):
                if (cand / "src" / "services").is_dir():
                    return cand.resolve()
    kin = Path("/kaggle/input")
    if kin.is_dir():
        for top in sorted(kin.iterdir()):
            if not top.is_dir():
                continue
            for cand in (top, top / NEST):
                if (cand / "src" / "services").is_dir():
                    return cand.resolve()
            try:
                for sub in top.iterdir():
                    if not sub.is_dir():
                        continue
                    for cand in (sub, sub / NEST):
                        if (cand / "src" / "services").is_dir():
                            return cand.resolve()
            except OSError:
                pass
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for cand in (base, base / NEST):
            if (cand / "src" / "services").is_dir():
                return cand.resolve()
    raise FileNotFoundError(
        "Không thấy src/services. Kaggle: Add Data (dataset chứa full repo có src/) "
        "hoặc !git clone ... vào /kaggle/working rồi %cd vào repo; "
        "hoặc set LOGIC_PROJECT_ROOT=/kaggle/input/.../Logic_Based_Educational_Queries_Project"
    )


R = logic_repo_root()
sys.path.insert(0, str(R / "src"))
from services.config import load_dotenv_for_logic

os.chdir(R / "notebooks")
print("PROJECT_ROOT:", R, "| cwd:", Path.cwd(), "| .env:", load_dotenv_for_logic())


## 3. Cấu hình — chỉnh tại đây

In [ ]:
from services.config import LogicSFTConfig

# Đọc `.env` (sao chép từ `.env.example` → `.env`). Ghi đè tại đây nếu cần.
cfg = LogicSFTConfig.from_env()
print("Config OK", cfg.model_name, cfg.data_source)
print("HF repo (push):", cfg.resolved_hf_repo())

## 4. Tải dữ liệu (chỉ khi `cfg.data_source == "drive"`)
Bỏ qua nếu dùng JSON trong repo (`local`).

In [ ]:
if cfg.data_source == "drive":
    get_ipython().run_line_magic("run", "-i logic_model_stage_drive.py")
else:
    print("DATA_SOURCE=local — bỏ qua tải Drive.")

## 5. Fine-tune (LoRA + chọn model theo accuracy trên dev)

In [ ]:
%run -i logic_model_stage_train.py

## 6. Push Hugging Face (tùy chọn)
Đặt `cfg.push_to_hub = True` và có `HF_Token`.

In [ ]:
%run -i logic_model_stage_push.py